# Day 015：进入 Full SFT 真实训练循环

本 Notebook 与 `day-015-2026-09-01.md` 配套，复现今天学习的训练主线：epoch、随机索引、batch 迭代、动态学习率、next-token loss、梯度累积、梯度裁剪、AdamW 更新、日志/保存边界。

恢复点：`trainer/train_full_sft.py` 的主循环下一步，以及训练结束后的分布式清理。

## 1. epoch 与随机索引

默认 `start_epoch=0`、`epochs=2` 时，`range` 产生 0、1；每个 epoch 原则上完整遍历一次数据。`setup_seed` 只设置随机起点，`randperm` 才生成不重复的 item 索引排列。

In [ ]:
import math
import random
import torch
import torch.nn.functional as F

epochs = list(range(0, 2))
random.seed(42)
indices = torch.randperm(5, generator=torch.Generator().manual_seed(42)).tolist()
print('epoch values：', epochs)
print('item indices：', indices)
assert epochs == [0, 1]
assert sorted(indices) == [0, 1, 2, 3, 4]
assert len(set(indices)) == 5

## 2. SkipBatchSampler 的完整批次和不足批次

50 个 item、batch_size=16 时，批次数为向上取整的 4；最后一批只有 2 个 item，但不会因为不满 16 而丢弃。

In [ ]:
def make_batches(indices, batch_size, skip_batches=0):
    batch = []
    skipped = 0
    for idx in indices:
        batch.append(idx)
        if len(batch) == batch_size:
            if skipped < skip_batches:
                skipped += 1
                batch = []
                continue
            yield batch
            batch = []
    if len(batch) > 0 and skipped >= skip_batches:
        yield batch

batches = list(make_batches(list(range(50)), 16))
print([len(b) for b in batches])
assert [len(b) for b in batches] == [16, 16, 16, 2]
skipped_batches = list(make_batches(list(range(50)), 16, skip_batches=2))
print('跳过两个 batch 后：', [len(b) for b in skipped_batches])
assert [len(b) for b in skipped_batches] == [16, 2]

## 3. next-token 对齐与 `-100`

原始位置 `t` 的 logits 预测位置 `t+1` 的 token，所以去掉 logits 最后一位、labels 第一位。`-100` 位置不参与交叉熵平均。

In [ ]:
logits = torch.tensor([[2.0, 1.0, 0.0], [0.0, 1.0, 2.0]])
targets = torch.tensor([0, -100])
ignored = F.cross_entropy(logits, targets, ignore_index=-100)
first_only = F.cross_entropy(logits[:1], torch.tensor([0]))
both = F.cross_entropy(logits, torch.tensor([0, 1]))
print('忽略第二行：', ignored)
print('只算第一行：', first_only)
print('两行都算：', both)
assert torch.allclose(ignored, first_only)
assert not torch.allclose(both, first_only)

toy_logits = torch.randn(2, 4, 3)
toy_labels = torch.tensor([[0, 1, 2, -100], [-100, 1, 0, -100]])
x = toy_logits[:, :-1, :].contiguous()
y = toy_labels[:, 1:].contiguous()
print('x shape：', tuple(x.shape), 'y shape：', tuple(y.shape))
print('flattened：', tuple(x.view(-1, x.size(-1)).shape), tuple(y.view(-1).shape))

## 4. 梯度裁剪与梯度累积

裁剪按整体范数统一缩放；梯度累积时每个微批次先除以 accumulation_steps，并在达到步数后更新和清零。

In [ ]:
from torch.nn.utils import clip_grad_norm_

p1 = torch.nn.Parameter(torch.tensor([3.0, 4.0]))
p2 = torch.nn.Parameter(torch.tensor([0.0, 12.0]))
p1.grad = torch.tensor([3.0, 4.0])
p2.grad = torch.tensor([0.0, 12.0])
before = torch.sqrt((p1.grad ** 2).sum() + (p2.grad ** 2).sum())
returned = clip_grad_norm_([p1, p2], max_norm=5.0)
after = torch.sqrt((p1.grad ** 2).sum() + (p2.grad ** 2).sum())
print('before：', before, 'returned：', returned, 'after：', after)
assert torch.allclose(before, torch.tensor(13.0))
assert torch.allclose(after, torch.tensor(5.0))

accumulation_steps = 4
updated_steps = [step for step in range(1, 11) if step % accumulation_steps == 0]
tail_exists = 10 % accumulation_steps != 0
print('循环内更新：', updated_steps, '尾部是否还有更新：', tail_exists)
assert updated_steps == [4, 8]
assert tail_exists is True

## 5. 训练末尾保存顺序的源码细节

当 `iters % accumulation_steps != 0` 时，循环体的 `step == iters` 保存发生在尾部 `scaler.step(optimizer)` 之前。下面用一个标量参数模拟：磁盘快照先保存旧值，随后内存参数才应用残余梯度。

In [ ]:
parameter = torch.nn.Parameter(torch.tensor(10.0))
parameter.grad = torch.tensor(2.0)
saved_value_before_tail_update = parameter.detach().clone()
with torch.no_grad():
    parameter -= 0.1 * parameter.grad  # 模拟第 79 行 scaler.step(optimizer)
print('保存时的值：', saved_value_before_tail_update.item())
print('尾部更新后的内存值：', parameter.item())
assert saved_value_before_tail_update.item() == 10.0
assert torch.allclose(parameter.detach(), torch.tensor(9.8))

## 6. `train_epoch()` 返回后，对象状态仍然保留

函数没有显式 `return` 时返回 `None`，但对传入对象的原地修改不会消失。真实训练中的模型参数、AdamW m/v 和 scaler 状态也是这样跨 epoch 保留。

In [ ]:
state = {'weight': 3.0, 'adam_m': 0.0}

def toy_train_epoch(external_state):
    external_state['weight'] = 4.0
    external_state['adam_m'] = 0.5

returned = toy_train_epoch(state)
print('函数返回：', returned)
print('外部对象：', state)
assert returned is None
assert state == {'weight': 4.0, 'adam_m': 0.5}

## 7. 用 4 条数据运行真实 Full SFT

先从原 JSONL 抽取 4 个完整 item，再从 `minimind/trainer` 运行训练。`learning_probe` 是专用实验名称，不会覆盖 `pretrain_768.pth` 或 `full_sft_768.pth`。

```bash
sed -n '1,4p' dataset/sft_t2t_mini.jsonl > dataset/sft_learning_4.jsonl
cd trainer
python train_full_sft.py \
  --save_dir ../out \
  --save_weight learning_probe \
  --epochs 1 --batch_size 1 --learning_rate 1e-5 \
  --device cpu --dtype bfloat16 --num_workers 0 \
  --accumulation_steps 1 --grad_clip 1.0 \
  --log_interval 1 --save_interval 1000 \
  --hidden_size 768 --num_hidden_layers 8 --max_seq_len 64 \
  --use_moe 0 --data_path ../dataset/sft_learning_4.jsonl \
  --from_weight pretrain --from_resume 0 --use_compile 0
```

本次真实日志的四个 loss 是 `6.4684、4.8482、3.5540、4.1988`，动态学习率是 `8.68e-6、5.50e-6、2.32e-6、1.00e-6`。四个 loss 来自不同样本，不能把它们直接当成同一题上的连续进步曲线。

## 8. 比较训练前后的真实参数文件

训练改变参数数值，不改变参数 shape。下面寻找 MiniMind 仓库并比较共享的 `model.embed_tokens.weight`。运行前需已经完成上面的最小训练。

In [ ]:
from pathlib import Path

def find_minimind_repo():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / 'minimind'):
            if (candidate / 'model' / 'model_minimind.py').exists():
                return candidate
    raise FileNotFoundError('没有找到 MiniMind 仓库目录')

repo = find_minimind_repo()
before_path = repo / 'out' / 'pretrain_768.pth'
after_path = repo / 'out' / 'learning_probe_768.pth'
assert before_path.exists() and after_path.exists(), '请先完成第 7 节的最小训练'

before = torch.load(before_path, map_location='cpu')
after = torch.load(after_path, map_location='cpu')
key = 'model.embed_tokens.weight'
diff = after[key].float() - before[key].float()
print('仓库：', repo)
print('shape：', tuple(before[key].shape), tuple(after[key].shape))
print('变化元素：', int((diff != 0).sum()), '/', diff.numel())
print('最大绝对变化：', diff.abs().max().item())
print('变化范数：', torch.linalg.vector_norm(diff).item())
assert before[key].shape == after[key].shape
assert diff.abs().max() > 0
del before, after, diff

## 9. 为什么两次 `dataset[0]` 可能不同

`SFTDataset.__getitem__()` 会以一定概率添加 system prompt，并以一定概率删除空的 `<think>`。因此索引相同不等于预处理结果必然相同。固定 Python 随机种子后，同一个索引才得到可复现的 token 和 labels。

In [ ]:
import sys
from transformers import AutoTokenizer

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
from dataset.lm_dataset import SFTDataset

tokenizer = AutoTokenizer.from_pretrained(repo / 'model')
dataset = SFTDataset(str(repo / 'dataset' / 'sft_learning_4.jsonl'), tokenizer, max_length=64)
random.seed(42)
ids_a, labels_a = dataset[0]
random.seed(42)
ids_b, labels_b = dataset[0]
print('input_ids 相同：', torch.equal(ids_a, ids_b))
print('labels 相同：', torch.equal(labels_a, labels_b))
print('有效 label 数：', int((labels_a != -100).sum()))
assert torch.equal(ids_a, ids_b)
assert torch.equal(labels_a, labels_b)

## 10. 训练时的真实前文与生成时的自身输出

训练时整条真实序列来自数据集，因果遮罩只限制每个位置不能看未来；生成时，新 token 会被拼回输入，下一轮会看到模型自己的上一轮输出。`x` 是 logits，不是 input token；`y` 才是错开一位后的正确 token ID。

In [ ]:
input_ids = torch.tensor([[10, 20, 30, 40]])
labels = input_ids.clone()
fake_logits = torch.zeros(1, 4, 6400)
x = fake_logits[..., :-1, :]
y = labels[..., 1:]
print('x shape：', tuple(x.shape))
print('y：', y.tolist())
assert x.shape == (1, 3, 6400)
assert y.tolist() == [[20, 30, 40]]

## 11. 训练后推理诊断

`learning_probe` 对“你是谁”连续生成 32 个 ID 234，而 ID 234 解码为换行 `\n`。它没有生成 EOS，所以直到 `max_new_tokens=32` 才停止。首 token 对比为：pretrain 的换行概率约 9.9%，learning_probe 的换行概率约 90.1%，完整 full_sft 的“我”概率约 88.5%。这说明训练、保存和加载链路成立，但 4 条样本的全参数训练产生了严重的小样本过拟合；这个实验只验证工程链路，不代表得到可用模型。

## Day 15 完成与下一恢复点

Full SFT 已从 epoch/DataLoader 一直闭环到真实训练、保存、重新加载和生成诊断。下一学习日从 `trainer/train_pretrain.py` 进入预训练，重点比较 `PretrainDataset` 与 `SFTDataset` 的输入和 labels，并解释预训练学习续写、SFT 学习按角色和指令回答。